In [ ]:
import os
import shutil
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input
from PIL import Image, UnidentifiedImageError
import pandas as pd
from sklearn.model_selection import train_test_split

# =========================
# CONFIG
# =========================
RAW_DATASET_DIR = Path("dataset\\wikiart")          # pasta original com as subpastas dos autores
CLEAN_DATASET_DIR = Path("dataset\\dataset_clean")  # pasta para guardar imagens válidas e convertidas
SPLIT_DATASET_DIR = Path("dataset\\dataset_split")  # pasta final com train/val/test

IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
MIN_WIDTH = 64
MIN_HEIGHT = 64

TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15

RANDOM_STATE = 42

assert abs(TRAIN_SIZE + VAL_SIZE + TEST_SIZE - 1.0) < 1e-6, "Os splits têm de somar 1."

Keeping only valid images and organising the dataset:
- `is_valid_image_file` checks if the extension is a valid format.
- `clean_dataset` checks if the files are corrupted, converts to RGB format, filters by size (64/64) and padronises it to .jpg

In [ ]:
def is_valid_image_file(file_path):
    return file_path.suffix.lower() in IMG_EXTENSIONS


def clean_dataset(raw_dir, clean_dir, min_width=64, min_height=64):
    clean_dir.mkdir(parents=True, exist_ok=True)

    removed_files = []
    kept_files = []

    class_dirs = [d for d in raw_dir.iterdir() if d.is_dir()]

    for class_dir in class_dirs:
        target_class_dir = clean_dir / class_dir.name
        target_class_dir.mkdir(parents=True, exist_ok=True)

        for file_path in class_dir.iterdir():
            if not file_path.is_file():
                continue

            if not is_valid_image_file(file_path):
                removed_files.append((str(file_path), "invalid"))
                continue

            try:
                with Image.open(file_path) as img:
                    img.verify()

                with Image.open(file_path) as img:
                    img = img.convert("RGB")

                    if img.width < min_width or img.height < min_height:
                        removed_files.append((str(file_path), f"image too small: {img.width}x{img.height}"))
                        continue

                    output_filename = file_path.stem + ".jpg"
                    output_path = target_class_dir / output_filename
                    img.save(output_path, format="JPEG", quality=95)

                    kept_files.append((str(output_path), class_dir.name, img.width, img.height))

            except (UnidentifiedImageError, OSError, Image.DecompressionBombError) as e:
                removed_files.append((str(file_path), f"error opening/processing: {e}"))

    kept_df = pd.DataFrame(kept_files, columns=["filepath", "label", "width", "height"])
    removed_df = pd.DataFrame(removed_files, columns=["filepath", "reason"])

    return kept_df, removed_df

In [ ]:
kept_df, removed_df = clean_dataset(
    raw_dir=RAW_DATASET_DIR,
    clean_dir=CLEAN_DATASET_DIR,
    min_width=MIN_WIDTH,
    min_height=MIN_HEIGHT
)

print("Valid:", len(kept_df))
print("Removed:", len(removed_df))


Imagens válidas: 13340
Imagens removidas: 0


Verify if all authors have an appropriate number of data.

In [7]:
class_counts = kept_df["label"].value_counts().sort_values(ascending=False)

print("\nNumber of images per author:")
print(class_counts)

summary_df = class_counts.reset_index()
summary_df.columns = ["author", "n_images"]

print("\nSummary:")
print(summary_df)


Number of images per author:
label
Vincent_van_Gogh         1322
Nicholas_Roerich         1274
Pierre_Auguste_Renoir     975
Claude_Monet              934
Pyotr_Konchalovsky        644
Camille_Pissarro          621
Albrecht_Durer            580
John_Singer_Sargent       549
Rembrandt                 544
Marc_Chagall              536
Pablo_Picasso             534
Gustave_Dore              528
Boris_Kustodiev           444
Edgar_Degas               428
Paul_Cezanne              406
Ivan_Aivazovsky           404
Martiros_Saryan           403
Eugene_Boudin             389
Childe_Hassam             385
Ilya_Repin                378
Ivan_Shishkin             364
Raphael_Kirchner          362
Salvador_Dali             336
Name: count, dtype: int64

Summary:
                   author  n_images
0        Vincent_van_Gogh      1322
1        Nicholas_Roerich      1274
2   Pierre_Auguste_Renoir       975
3            Claude_Monet       934
4      Pyotr_Konchalovsky       644
5        Camille_Pissa

In [8]:
def create_splits(df, train_size=0.70, val_size=0.15, test_size=0.15, random_state=42):
    train_df, temp_df = train_test_split(
        df,
        test_size=(1 - train_size),
        stratify=df["label"],
        random_state=random_state
    )

    val_relative = val_size / (val_size + test_size)

    val_df, test_df = train_test_split(
        temp_df,
        test_size=(1 - val_relative),
        stratify=temp_df["label"],
        random_state=random_state
    )

    return train_df, val_df, test_df

In [9]:
train_df, val_df, test_df = create_splits(
    kept_df,
    train_size=TRAIN_SIZE,
    val_size=VAL_SIZE,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nDistribuição train:")
print(train_df["label"].value_counts())

print("\nDistribuição val:")
print(val_df["label"].value_counts())

print("\nDistribuição test:")
print(test_df["label"].value_counts())

Train: 9337
Validation: 2001
Test: 2002

Distribuição train:
label
Vincent_van_Gogh         925
Nicholas_Roerich         892
Pierre_Auguste_Renoir    682
Claude_Monet             654
Pyotr_Konchalovsky       451
Camille_Pissarro         435
Albrecht_Durer           406
John_Singer_Sargent      384
Rembrandt                381
Marc_Chagall             375
Pablo_Picasso            374
Gustave_Dore             369
Boris_Kustodiev          311
Edgar_Degas              300
Paul_Cezanne             284
Ivan_Aivazovsky          283
Martiros_Saryan          282
Eugene_Boudin            272
Childe_Hassam            269
Ilya_Repin               265
Ivan_Shishkin            255
Raphael_Kirchner         253
Salvador_Dali            235
Name: count, dtype: int64

Distribuição val:
label
Vincent_van_Gogh         198
Nicholas_Roerich         191
Pierre_Auguste_Renoir    146
Claude_Monet             140
Pyotr_Konchalovsky        96
Camille_Pissarro          93
Albrecht_Durer            87
John_Singer_

In [10]:
def copy_files(df, split_name, base_dir):
    for _, row in df.iterrows():
        src = row["filepath"]
        label = row["label"]

        # criar pasta destino
        dst_dir = os.path.join(base_dir, split_name, label)
        os.makedirs(dst_dir, exist_ok=True)

        # copiar ficheiro
        dst = os.path.join(dst_dir, os.path.basename(src))
        shutil.copy(src, dst)

copy_files(train_df, "train", SPLIT_DATASET_DIR)
copy_files(val_df, "val", SPLIT_DATASET_DIR)
copy_files(test_df, "test", SPLIT_DATASET_DIR)

`image_dataset_from_directory` - load and label images from training, validation, and test folders, standardizing them to a uniform $224 \times 224$ pixel resolution in batches of 32.

`preprocess_input function` - scale and normalization of pixel values to match the specific mathematical requirements.

`AUTOTUNE` and `prefetch` - CPU prepares the next batch of data while the GPU is still processing the current one, eliminating hardware bottlenecks and significantly speeding up the training process.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    SPLIT_DATASET_DIR / "train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    SPLIT_DATASET_DIR / "val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    SPLIT_DATASET_DIR / "test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)

train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
test_ds = test_ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)